In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(1)

rows = []
for part in range(1, 11):
    part_eff = rng.normal(0, 1.0)
    for op in ["A", "B", "C"]:
        op_eff = rng.normal(0, 0.25)
        for station in ["S1", "S2"]:
            station_eff = rng.normal(0, 0.15)
            for trial in [1, 2, 3]:
                y = 10 + part_eff + op_eff + station_eff + rng.normal(0, 0.2)
                rows.append([part, op, station, trial, y])

df = pd.DataFrame(rows, columns=["Part", "Operator", "Station", "Trial", "Value"])

In [4]:
#!pip install statsmodels
import statsmodels.api as sm

model = sm.MixedLM.from_formula(
    "Value ~ 1",
    groups="Part",
    re_formula="1",
    vc_formula={
        "Operator": "0 + C(Operator)",
        "Station": "0 + C(Station)",
    },
    data=df,
)

result = model.fit(reml=True)
print(result.summary())
print("Residual variance:", result.scale)
print("Groups variance:", result.cov_re)
print("Variance components:", result.vcomp)


         Mixed Linear Model Regression Results
Model:             MixedLM Dependent Variable: Value   
No. Observations:  180     Method:             REML    
No. Groups:        10      Scale:              0.0401  
Min. group size:   18      Log-Likelihood:     -21.2166
Max. group size:   18      Converged:          Yes     
Mean group size:   18.0                                
-------------------------------------------------------
             Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------
Intercept    10.083    0.234 43.047 0.000  9.623 10.542
Part Var      0.524    1.328                           
Operator Var  0.053    0.100                           
Station Var   0.010    0.033                           

Residual variance: 0.040137961263216046
Groups variance:           Part
Part  0.523917
Variance components: [0.0527349  0.00975381]


/home/henry/miniforge3/envs/gagerr/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
